[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://drive.google.com/file/d/19B_7d6jcB_slTPzhOa7p-c2LL2osiM_1/view?usp=drive_link)

# Agent Evaluation – FloTorch Hosted Agent

This notebook demonstrates how to evaluate an agent deployed on the FloTorch gateway. Floeval sends each test case to the gateway, the hosted agent processes it, and Floeval captures the trace and scores the result.

**FloTorch Console:** [https://docs.flotorch.cloud/introduction/](https://docs.flotorch.cloud/introduction/)

**Prerequisites**
- Agent deployed in the [FloTorch Console](https://docs.flotorch.cloud/introduction/)
- `pip install floeval[flotorch]`
- FloTorch API key and gateway URL

**Objectives**
- Install Floeval with FloTorch support and configure credentials
- Build a partial agent dataset with test cases
- Create a FloTorch runner with `create_flotorch_runner`
- Run `AgentEvaluation` with the runner and inspect results

## 1. Installation

Install Floeval with FloTorch support. Required for evaluating agents deployed on the FloTorch gateway.

In [ ]:
%pip install floeval[flotorch]>=0.2.0b1

## 2. Configuration Constants

Set the following constants before running. Obtain credentials and agent names from the [FloTorch Console](https://docs.flotorch.cloud/introduction/).

**Provider flexibility:** You can use any OpenAI-compatible provider. For FloTorch-hosted agents, use the FloTorch gateway URL and keys from the [FloTorch Console](https://docs.flotorch.cloud/introduction/).

In [ ]:
import getpass

# FloTorch gateway configuration
FLOTORCH_BASE_URL = "https://gateway.flotorch.cloud/openai/v1"
FLOTORCH_API_KEY = getpass.getpass("your-flotorch-api-key")
FLOTORCH_CHAT_MODEL =  "<flotorch-model>"
FLOTORCH_AGENT_NAME = "<agent-name>"
FLOTORCH_EMBEDDING_MODEL = "<flotorch-embedding-model>"

## 1. Imports

The following cell imports the agent evaluation components, the dataset schemas, and `create_flotorch_runner` from the flotorch module.

In [ ]:
from floeval.api.agent_evaluation import AgentEvaluation
from floeval.config.schemas.io.agent_dataset import AgentDataset, PartialAgentSample, ToolCall
from floeval.config.schemas.io.llm import OpenAIProviderConfig
from floeval.flotorch import create_flotorch_runner

## 2. Configure the LLM (FloTorch Gateway)

The LLM configuration is built for the FloTorch gateway. Set `FLOTORCH_BASE_URL` and `FLOTORCH_API_KEY` in your environment or pass them explicitly.

In [ ]:
llm_config = OpenAIProviderConfig(
    base_url=FLOTORCH_BASE_URL,
    api_key=FLOTORCH_API_KEY,
    chat_model=FLOTORCH_CHAT_MODEL,
    embedding_model=FLOTORCH_EMBEDDING_MODEL,
)

## 3. Create the FloTorch Runner

`create_flotorch_runner` connects to your deployed agent by name. Replace `my-agent` with your actual agent name from the FloTorch Console.

In [ ]:
agent_name = FLOTORCH_AGENT_NAME
runner = create_flotorch_runner(agent_name, llm_config=llm_config)
print(f"Runner created for agent: {agent_name}")

## 4. Build the Partial Dataset

A partial agent dataset is built with test cases containing `user_input` and optionally `reference_outcome` and `reference_tool_calls`.

In [ ]:
dataset = AgentDataset(
    samples=[
        PartialAgentSample(
            user_input="What is the weather in Tokyo?",
            reference_outcome="Provides Tokyo weather",
            reference_tool_calls=[ToolCall(name="search_tool")],
        )
    ]
)
print(f"Partial dataset loaded: {len(dataset.samples)} sample(s)")

## 5. Create and Run the Evaluation

The `agent_runner=runner` parameter is passed to `AgentEvaluation`. Floeval sends each sample to the gateway, the hosted agent runs, and Floeval captures the trace and scores it.

In [ ]:
evaluation = AgentEvaluation(
    dataset=dataset,
    agent_runner=runner,
    llm_config=llm_config,
    metrics=["goal_achievement", "response_coherence", "ragas:tool_call_accuracy"],
    default_provider="builtin",
)

results = evaluation.run()
print("Summary:", results.summary)

## Summary

This notebook demonstrated the evaluation of agents deployed on the FloTorch gateway.

The key components included:

1. **LLM Configuration**: The FloTorch gateway URL and API key were configured for the evaluation.
2. **FloTorch Runner**: A runner was created with `create_flotorch_runner(agent_name, llm_config)` to connect to the deployed agent.
3. **Partial Dataset**: A partial agent dataset was built with test cases.
4. **Evaluation Execution**: The `agent_runner` parameter was passed to `AgentEvaluation` for gateway-based execution.
5. **Metrics**: The `goal_achievement`, `response_coherence`, and `tool_call_accuracy` metrics were run.

This example showcases the workflow for evaluating FloTorch-hosted agents.